# Lab 4b: Animation Graphs & Motion Matching
**Course:** CIS 536/736: Computer Graphics
**Date:** Friday, October 23, 2026 (Week 9)
**Associated Deliverable:** PS 6 (Rigging & Animation Graph Design)

This lab bridges the theoretical kinematics and graph design from PS 6 into a functional Unity 6 implementation. You will transition from rigid, deterministic state machines to a data-driven Motion Matching solver, culminating in a hybrid character controller.

## Part 1: Asset Preparation & Humanoid Retargeting
Before applying any animation logic, your skeletal hierarchy must be standardized so the solver can map the raw MoCap database to your mesh.

1. **Import Settings:** Import your character `.fbx` and the provided raw locomotion MoCap dataset.
2. **Rig Integrity:** In the Unity Inspector, set the Animation Type to **Humanoid** and configure the Avatar. Verify that all required bone mappings (especially the spine and leg chains) are assigned correctly.
3. **Root Motion Alignment:** Ensure the root node's translation and rotation are baked correctly. If the root transform does not drive the character in world space, the solver will cause foot sliding.

## Part 2: Baseline State Machine Construction
To verify rig integrity, we first build a traditional deterministic state machine (as covered in Lecture 22). This will later serve as the fallback for discrete actions in our hybrid system.

1. Create a new **Animator Controller**.
2. Build a 2D Freeform Directional **Blend Tree** for your base locomotion.
3. Bind the `Speed` and `Direction` float parameters.

In [ ]:
using UnityEngine;

// Baseline C# Input Script to drive the standard Animator Blend Tree
public class BaselineLocomotion : MonoBehaviour
{
    private Animator animator;
    
    void Start()
    {
        animator = GetComponent<Animator>();
    }

    void Update()
    {
        float inputX = Input.GetAxis("Horizontal");
        float inputY = Input.GetAxis("Vertical");
        
        Vector2 inputVector = new Vector2(inputX, inputY);
        animator.SetFloat("Speed", inputVector.magnitude);
        animator.SetFloat("Direction", inputX);
    }
}

## Part 3: Motion Matching Setup & The Cost Function
Now we replace the base locomotion layer with the Unity 6 Motion Matching solver. The solver replaces explicit transitions with a continuous database query to find the mathematically optimal next frame.

### 3.1 Pose Database Generation
Add your MoCap clips to a **Motion Sequence** asset and bake the database. Unity will extract key features (root, feet, hips) and build an optimized Kd-Tree for real-time querying. Use **Data Tagging** (e.g., tagging crouch vs. sprint frames) to restrict the search space and prevent physically invalid jumps.

### 3.2 Tuning the Cost Function
The solver minimizes the cost function: `C = (W_p * P) + (W_v * V) + (W_t * T)`
* **P (Pose Alignment):** Matches the current skeletal state (e.g., left foot planted).
* **V (Bone Velocity):** Matches the current momentum of the limbs.
* **T (Trajectory):** Matches the player's future inputted path.

In the Motion Matching component, tune the weights (`W_p`, `W_v`, `W_t`). High trajectory weighting causes skating; high pose weighting causes unresponsiveness.

In [ ]:
using UnityEngine;
using UnityEngine.Animations.MotionMatching;

// C# Script to drive the Trajectory Generator for Motion Matching
public class TrajectoryDriver : MonoBehaviour
{
    private TrajectoryGenerator trajectoryGenerator;
    public Transform cameraTransform;

    void Start()
    {
        trajectoryGenerator = GetComponent<TrajectoryGenerator>();
    }

    void Update()
    {
        // Project past, present, and future trajectory points (e.g., 0.5s, 1.0s, 1.5s)
        Vector3 moveInput = new Vector3(Input.GetAxis("Horizontal"), 0, Input.GetAxis("Vertical"));
        Vector3 worldMove = cameraTransform.TransformDirection(moveInput);
        worldMove.y = 0;
        
        // Feed the predicted future path to the Motion Matching solver
        trajectoryGenerator.SetDesiredTrajectory(worldMove.normalized);
    }
}

## Part 4: Hybrid System Integration & Avatar Masks
Motion Matching cannot handle discrete, deterministic actions (like swinging a sword or opening a door). We must construct a **Hybrid System**.

1. Keep the **Motion Matching Solver** on the Base Layer of your Animator.
2. Create a new Animator Layer for discrete actions.
3. Apply an **Avatar Mask** to the upper body. This allows the deterministic state machine to control the arms/spine (e.g., aiming) while the Motion Matching solver fluidly drives the lower body locomotion.

## Part 5: Debugging & Polish
Debugging a data-driven system requires runtime visualizers, not static node graphs.

* **Enable Debug Gizmos:** Turn on the Motion Matching debug view in the Scene tab to render the projected trajectory spline and candidate poses.
* **Fixing Transition Popping:** If the character snaps unnaturally, pause the editor and read the cost graph. Determine whether a trajectory mismatch or a missing pose in the database triggered the jump.
* **Responsiveness Polish:** Adjust the trajectory projection window (e.g., shortening the look-ahead from 1.0s to 0.5s) to make input feel tighter and less sluggish.